# Day 1 -> Day 3 SIC: full nested cross-validation pipeline

**Input**: `data/NE_baseline_D1.csv` (99 patients, Day 1) + `data/NE_baseline_D3.csv` (Day 3 SIC)
**Output**: Figure 3, Supplementary Figures S6/S7, sTable S3-S7 data, result pkl/CSV
**Run**: execute all cells from top to bottom

Two methodological points (in response to reviewers):
1. **Feature selection and performance estimation are kept separate** - the outer 5-fold x 20
   repeats are used for evaluation only; imputation / standardisation / tuning / backward
   elimination / threshold selection are re-run inside every outer training fold, and the test
   fold never takes part in any modelling decision;
2. **Statistical inference** - paired comparisons use the Nadeau-Bengio corrected t-test (df = 19),
   and 95% CIs use patient-level bootstrap (B = 2000, resampling pooled per-patient OOF predictions).

## Pipeline overview

**In one sentence**: the outer 5-fold x 20 repeats are used **for evaluation only**; imputation,
standardisation, tuning, feature ranking, backward elimination and threshold selection are all
re-done **inside each outer training fold**; the test fold is predicted once and never informs
any modelling decision.

```text
NE_baseline_D1.csv + NE_baseline_D3.csv
|-- intersect on Code -> 99 patients (SIC 40 / NSIC 59); y = Day-3 ISTH SIC >= 4
   |-- outer RepeatedStratifiedKFold (5 folds x 20 repeats = 100 folds)
      |-- training fold (~79 patients, ~32 events)
      |  |-- fixed models x3: Clinical_Only(7) / Single_Immature(1) / Clinical_Immature(8)
      |  |     fit_pipeline_on_train()
      |  |        IterativeImputer -> StandardScaler
      |  |        -> RandomizedSearchCV(elastic-net LR; 12 C x 6 l1_ratio = 72 settings, cv=5)
      |  |        -> inner-OOF Youden threshold
      |  |-- RFECV (simplified model) branch
      |        |-- rfecv_selection()          inner CV inside the training fold (5 folds, ~63/16)
      |        |  |-- fixed pipeline: impute -> standardise -> elastic-net LR (C=1, l1_ratio=1)
      |        |  |-- recursive elimination from 8 down to 3 predictors, inner-CV AUROC per size
      |        |  |-- keep the size with the highest inner-CV AUROC; floor MIN_FEATURES=3
      |        |-- fit_pipeline_on_train(selected features) -> inner-OOF Youden threshold
      |-- test fold (~20 patients) - predict_proba exactly once
   |-- pooled OOF: combine the 5 folds per repeat -> average 20 repeats per patient -> pooled estimate
      |-- patient-level bootstrap 95% CI (B=2000, resample (y, pooled probability) with replacement)
         |-- paired comparison: both models scored within the same resample (pooled paired bootstrap)
            |-- sensitivity: Nadeau-Bengio corrected t (repeat level, df=19) -> separate CSV
            |-- non-inferiority: lower 95% CI bound of dAUROC > -0.05
   |-- Part 2 (execution order = narrative order)
      |  |-- (2) four models + paired comparisons (sTable S3-S4)
      |  |-- (3) single-marker models (sTable S5; same pipeline as S3, not raw-value ROC)
      |  |-- (4) four algorithms (sTable S6 / Fig S6; input = all 8 predictors)
      |  |-- (5) NSIC subgroup (sTable S7, descriptive only)
   |-- Part 3: figures (Figure 3/4, Supplementary Figures S6/S7)
```

### Division of labour across the three parts

| Part | What it does | Runtime (1 core / multi-core) | Products |
|---|---|---|---|
| **Part 1** | Outer nested CV: per-fold probabilities and thresholds for the four models | ~30 / 12 min | `nested_cv_results.pkl`, `nested_selection_log.csv` |
| **Part 2** | (1) full-data derivation -> (2) S4-S5 -> (3) single marker -> (4) four algorithms (ALL8) -> (5) subgroup | ~67 / 20 min | `nested_cv_stage2.pkl`, `backward_elim*`, `non_inferiority_3models.csv`, `table_s5_nadeau_bengio.csv` |
| **Part 3** | Figures from the pickles | ~1 min | `Figure 3/4`, `Supplementary Figures S6/S7` (tif + png + pdf) |

### Two nested loops

| Level | Split | What happens inside | Purpose |
|---|---|---|---|
| Outer | 5 folds x 20 repeats (`SEED=42`) | fixed models are tuned and fitted directly; the parsimonious model is selected first, then fitted | evaluation only; produces OOF probabilities |
| Inner | 5 folds x 3 repeats (`inner_cv_eval`) | imputation / standardisation / tuning re-done in every inner fold (`n_iter=12, cv=3`), returning per-fold AUROC and \|coef\| | rank importance and decide whether a predictor can be dropped |

### Key statistical conventions

| Item | Approach |
|---|---|
| Point estimate (pooled OOF) | **ensemble-of-repeated-CV convention**: each patient's 20 OOF predictions are averaged, equivalent to treating the 20 repeated fits as one ensemble; AUC/AUPRC/Brier are computed from it |
| 95% CI | patient-level bootstrap (B=2000), 2.5th/97.5th percentiles |
| Paired comparison | **pooled paired bootstrap**: both models are scored in the same resample before subtraction (S4/S5/non-inferiority share this convention) |
| Sensitivity analysis | Nadeau-Bengio corrected t is only an approximation (repeat level, near-independence assumed) and is reported as a sensitivity analysis -> `table_s5_nadeau_bengio.csv` |
| Multiplicity | 3 comparisons x 5 metrics = 15 tests; the study is **exploratory**, so Benjamini-Hochberg FDR (q values) is primary, with raw P and Holm also given |
| Non-inferiority | met if the lower 95% CI bound of the pooled dAUROC exceeds -0.05 |
| Operating-point metrics | Sen/Spec/PPV/NPV/F1 use the **inner-OOF Youden threshold of each fold** to classify the test fold, averaged over repeats; the test fold never informs the threshold |
| NSIC subgroup | **descriptive only** (n, events, median/IQR of predicted risk), no AUC |

> Implementation note: every candidate feature set and the final model fit the imputer/scaler on
> the **selected features only**; all LogisticRegression fits share `max_iter=LR_MAX_ITER (5000)`
> and `tol=1e-4`, and non-converged fits are counted in `CONVERGENCE_FAILURES` and printed at runtime.

> The full version (with a Mermaid diagram) is in [`pipeline.md`](pipeline.md).

## Imports

In [ ]:
import os
import sys
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             recall_score, precision_score, f1_score,
                             confusion_matrix, brier_score_loss)
from scipy.stats import uniform, loguniform, randint, t as t_dist, ttest_rel, ttest_ind, ttest_ind_from_stats

## Paths and global settings

In [ ]:
# ------------------------------------------------------------------ paths
try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # Jupyter environment
    HERE = os.getcwd()
ROOT = HERE
for _ in range(5):
    if os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'outputs')):
        break
    ROOT = os.path.dirname(ROOT)

if not (os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'outputs'))):
    raise RuntimeError(
        'Package root not found (should contain data/ and outputs/): current directory = %s.'
        'Start Jupyter inside NE_99_261022_EN, or cd into it first.' % ROOT)
DATA = os.path.join(ROOT, 'data')
OUT_T = os.path.join(ROOT, 'outputs', 'tables')
os.makedirs(OUT_T, exist_ok=True)

N_JOBS = int(os.environ.get('NE_JOBS', '-1'))   # export NE_JOBS=1 in sandboxes/CI
N_SPLITS = 5          # outer/inner folds
N_REPEATS = 20        # outer repeats
N_INNER_REPEATS = 3   # inner repeats (selection)
N_BOOT = 2000
LR_MAX_ITER = 5000      # shared by all LogisticRegression fits (saga+elasticnet needs many iterations)
LR_TOL = 1e-4
LOCK_BIOMARKER = False  # CD10- not forced: all 8 candidates compete equally
SELECTOR = 'rfecv'      # inner feature selection: RFECV (recursive elimination + inner-CV choice of size)
MIN_FEATURES = 3         # lower bound on subset size
RFECV_CV = 5             # inner CV folds for RFECV
FINAL_K = 4             # predictors in the final simplified model: top K by selection frequency
STOP_AT_PEAK = False     # False = keep eliminating until significantly worse (p<0.01) or the floor; keep the simplest non-inferior subset;
                         # True  = stop at the AUROC peak (most conservative, keeps more predictors)
CONVERGENCE_FAILURES = []   # records non-converged fits
SEED = 42
MARGIN = -0.05        # non-inferiority margin

## 1. Data (99 patients / 40 Day-3 SIC events)

In [ ]:
d1 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D1.csv'), encoding='utf-8-sig')
d3lab = pd.read_csv(os.path.join(DATA, 'NE_baseline_D3.csv'), encoding='utf-8-sig')
d3lab = d3lab[['Code', 'SIC']].dropna().set_index('Code')   # SIC = ISTH SIC score

d1 = d1[d1['Code'].isin(d3lab.index)].sort_values('Code').reset_index(drop=True)
y = (d3lab.loc[d1['Code'], 'SIC'].values == 1).astype(int)   # Day-3 SIC (score >= 4, binary)
print(f'analysis set n = {len(y)}  (Day-3 SIC {int(y.sum())} / NSIC {int((1 - y).sum())})')

d1['SOFA_Creatinine'] = d1['SOFA D1 Creatinine']   # renal SOFA uses creatinine only (urine output unreliable)
clinical_cols = ['INR', 'PLT', 'lac', 'SOFA D1 Circulation',
                 'SOFA D1 Respiratory System', 'VT_D1', 'SOFA_Creatinine']       # 7 clinical predictors
biomarker_col = 'immature_Neu_percentage'
ALL8 = clinical_cols + [biomarker_col]                                            # 8 predictors: clinical + single biomarker
X_raw_all = d1[ALL8].copy()
codes = d1['Code'].values

DISPLAY = {
    'INR': 'INR', 'PLT': 'Platelet count', 'lac': 'Lactate',
    'SOFA D1 Circulation': 'SOFA cardiovascular subscore',
    'SOFA D1 Respiratory System': 'SOFA respiratory subscore',
    'VT_D1': 'Mechanical ventilation', 'SOFA_Creatinine': 'SOFA renal subscore',
    'immature_Neu_percentage': 'CD10- neutrophils,%',
}

# narrowed search: C 0.01-10 (12 values) x l1_ratio 0.1-1.0 (6 values) = 72 settings
param_dist = {
    'C': np.logspace(-2, 1, 12),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
}

## 2. Building blocks (all fitted inside the training fold only)

### `fit_imputer_scaler()` - imputation and standardisation used by the pipeline

In [ ]:
def fit_imputer_scaler(Xtr_df):
    imp = IterativeImputer(max_iter=100, random_state=0).fit(Xtr_df) # MICE: 100 iterations, single imputation
    sc = StandardScaler().fit(imp.transform(Xtr_df)) # prevents data leakage
    return imp, sc

### `transform()`

In [ ]:
def transform(imp, sc, X_df):
    return sc.transform(imp.transform(X_df))

### `tune_lr()` - elastic-net hyperparameter tuning (inner CV search on the given data).

In [ ]:
def check_convergence(model, tag=''):
    """Record fits that hit the iteration cap; return whether converged."""
    n_it = getattr(model, 'n_iter_', None)
    if n_it is None:
        return True
    n_it = int(np.max(n_it))
    if n_it >= LR_MAX_ITER:
        CONVERGENCE_FAILURES.append((tag, n_it))
        return False
    return True



In [ ]:
def tune_lr(X, yv, seed, n_iter=20, cv=5):
    """Elastic-net hyperparameter tuning (CV search on the given data).

    A common iteration cap LR_MAX_ITER / LR_TOL is applied (saga + elasticnet often needs thousands
    of iterations on small samples; non-converged fits are recorded in CONVERGENCE_FAILURES).
    """
    s = RandomizedSearchCV(
        LogisticRegression(penalty='elasticnet', solver='saga',
                           max_iter=LR_MAX_ITER, tol=LR_TOL, random_state=SEED),
        param_dist, n_iter=n_iter, cv=cv, scoring='roc_auc', random_state=seed, n_jobs=N_JOBS)
    s.fit(X, yv)
    check_convergence(s.best_estimator_, tag='tune_lr(seed=%s)' % seed)
    return s.best_estimator_

### `youden_cutoff()`

In [ ]:
def youden_cutoff(yv, p):
    fpr, tpr, thr = roc_curve(yv, p)
    return thr[np.argmax(tpr - fpr)] if len(thr) else 0.5

### `inner_cv_eval()` - inner evaluation of a feature subset: repeated stratified 5-fold,
with imputation / standardisation / tuning re-done inside every inner fold.

In [ ]:
def inner_cv_eval(Xtr_df, ytr, features, seed, n_repeats=N_INNER_REPEATS):
    """Inner evaluation of a feature subset: repeated stratified 5-fold, with imputation, standardisation
    in each fold (n_iter=12, cv=3); returns per-fold AUROC and |coef|."""
    Xs = Xtr_df[list(features)]
    rskf_in = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=n_repeats,
                                      random_state=seed)
    aucs, coefs = [], []
    Xv = Xs.values
    for fi, (tr, te) in enumerate(rskf_in.split(Xv, ytr)):
        imp, sc = fit_imputer_scaler(Xs.iloc[tr])
        Xtr = sc.transform(imp.transform(Xs.iloc[tr]))
        Xte = sc.transform(imp.transform(Xs.iloc[te]))
        m = tune_lr(Xtr, ytr[tr], seed=seed * 1000 + fi, n_iter=12, cv=3)
        aucs.append(roc_auc_score(ytr[te], m.predict_proba(Xte)[:, 1]))
        coefs.append(np.abs(m.coef_[0]))
    return np.array(aucs), np.array(coefs)

In [ ]:
def make_lr_pipeline(C=1.0, l1_ratio=1.0):
    """Fixed impute -> standardise -> elastic-net LR pipeline for RFE (no outer tuning)."""
    from sklearn.pipeline import Pipeline
    return Pipeline([
        ('imp', IterativeImputer(max_iter=100, random_state=0)),
        ('sc', StandardScaler()),
        ('lr', LogisticRegression(penalty='elasticnet', solver='saga', C=C,
                                  l1_ratio=l1_ratio, max_iter=LR_MAX_ITER,
                                  tol=LR_TOL, random_state=SEED)),
    ])


def rfecv_selection(Xtr_df, ytr, seed):
    """Inner feature selection: RFECV (recursive elimination + inner-CV AUROC to choose the size).

    - fixed pipeline: impute -> standardise -> elastic-net LR (C=1, l1_ratio=1);
    - one predictor removed per step, from 8 down to MIN_FEATURES (3);
    - each size scored by inner-CV (stratified 5-fold) AUROC; the highest mean wins;
    - performed entirely inside the outer training fold; the test fold is untouched.
    Returns: (selected features, best size, {size: mean inner-CV AUROC})
    """
    from sklearn.feature_selection import RFECV
    from sklearn.model_selection import StratifiedKFold
    Xs = Xtr_df[ALL8]
    cv = StratifiedKFold(n_splits=RFECV_CV, shuffle=True, random_state=seed)
    rfecv = RFECV(estimator=make_lr_pipeline(), step=1,
                  min_features_to_select=MIN_FEATURES, cv=cv, scoring='roc_auc',
                  importance_getter='named_steps.lr.coef_')
    rfecv.fit(Xs, ytr)
    feats = [f for f, keep in zip(ALL8, rfecv.support_) if keep]
    cvres = rfecv.cv_results_
    curve = {int(k): float(s) for k, s in zip(cvres['n_features'], cvres['mean_test_score'])}
    return feats, int(rfecv.n_features_), curve

### `fit_pipeline_on_train()` - on the outer training set: fit imputer+scaler -> tune -> fit -> Youden threshold.

In [ ]:
def youden_cutoff_oof(Xs, ytr, tuned_model, seed, n_splits=5):
    """Derive the Youden threshold from inner-CV **out-of-fold** predictions.

    The impute -> standardise -> tuned-LR steps are wrapped in a Pipeline and cross-validated
    (stratified 5-fold) inside the training fold, giving one OOF probability per patient;
    the Youden optimum is then taken on these predictions.

    Note: hyperparameters were tuned on the whole training fold (tune_lr), so second-order
    optimism remains, but the threshold itself is now inner-OOF rather than apparent.
    """
    from sklearn.base import clone
    from sklearn.impute import IterativeImputer
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    pipe = Pipeline([
        ('imp', IterativeImputer(max_iter=100, random_state=0)),
        ('sc', StandardScaler()),
        ('lr', clone(tuned_model)),
    ])
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    p_oof = cross_val_predict(pipe, Xs, ytr, cv=cv, method='predict_proba')[:, 1]
    return youden_cutoff(ytr, p_oof)


def fit_pipeline_on_train(Xtr_df, ytr, features, seed):
    """Fit imputer / scaler / model / Youden threshold on the **selected features** only.

    - dropped columns take no part in imputation or scaling, so inputs match the model;
    - the threshold comes from inner-OOF predictions (not fitted values); see youden_cutoff_oof.
    """
    Xs = Xtr_df[list(features)].copy()
    imp, sc = fit_imputer_scaler(Xs)
    Xtr = sc.transform(imp.transform(Xs))
    m = tune_lr(Xtr, ytr, seed=seed)
    cut = youden_cutoff_oof(Xs, ytr, m, seed=seed)
    return {'imp': imp, 'sc': sc, 'features': list(features), 'model': m, 'cutoff': cut}

### `predict_bundle()`

In [ ]:
def predict_bundle(bundle, Xte_df):
    Xte = bundle['sc'].transform(bundle['imp'].transform(Xte_df[bundle['features']]))
    return bundle['model'].predict_proba(Xte)[:, 1]

## 3. Metrics and statistical inference

### `metrics_from_probs()`

In [ ]:
def metrics_from_probs(yv, p, cutoff):
    out = {'auc': roc_auc_score(yv, p), 'auprc': average_precision_score(yv, p),
           'brier': brier_score_loss(yv, p)}
    yp = (p >= cutoff).astype(int)
    out['sensitivity'] = recall_score(yv, yp)
    out['ppv'] = precision_score(yv, yp, zero_division=0)
    out['f1'] = f1_score(yv, yp, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(yv, yp, labels=[0, 1]).ravel()
    out['specificity'] = tn / (tn + fp) if (tn + fp) else 0.0
    out['npv'] = tn / (tn + fn) if (tn + fn) else 0.0
    return out

### `metrics_from_binary()` - discrimination metrics come from the continuous probability p; classification metrics come from the binary decision ybin

In [ ]:
def metrics_from_binary(yv, p, ybin):
    """Discrimination metrics come from the continuous probability p; classification metrics from ybin
    (in nested CV, ybin comes from each fold's training threshold; no test-fold information)."""
    yv = np.asarray(yv); p = np.asarray(p); ybin = np.asarray(ybin)
    out = {'auc': roc_auc_score(yv, p), 'auprc': average_precision_score(yv, p),
           'brier': brier_score_loss(yv, p)}
    tp = int(((ybin == 1) & (yv == 1)).sum()); fn = int(((ybin == 0) & (yv == 1)).sum())
    tn = int(((ybin == 0) & (yv == 0)).sum()); fp = int(((ybin == 1) & (yv == 0)).sum())
    out['sensitivity'] = tp / (tp + fn) if (tp + fn) else 0.0
    out['specificity'] = tn / (tn + fp) if (tn + fp) else 0.0
    out['ppv'] = tp / (tp + fp) if (tp + fp) else 0.0
    out['npv'] = tn / (tn + fn) if (tn + fn) else 0.0
    out['f1'] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    return out

### `bootstrap_binary_ci()` - patient-level bootstrap (binary-decision version): resample (y, pooled probability, pooled decision) at the patient level

In [ ]:
def bootstrap_binary_ci(yv, p, ybin, B=N_BOOT, seed=SEED):
    """Patient-level bootstrap (binary-decision version): resample (y, pooled probability, pooled decision)
    with replacement at the patient level; percentile 95% CI."""
    rng = np.random.default_rng(seed)
    keys = ['auc', 'auprc', 'sensitivity', 'specificity', 'ppv', 'npv', 'f1', 'brier']
    acc = {k: [] for k in keys}
    n = len(yv)
    for _ in range(B):
        ii = rng.integers(0, n, n)
        yb = yv[ii]
        if len(np.unique(yb)) < 2:
            continue
        m = metrics_from_binary(yb, p[ii], ybin[ii])
        for k in keys:
            acc[k].append(m[k])
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
            for k, v in acc.items()}

### `resummarize_oof()` - summary under the pooled-OOF convention (point estimate + interval):

In [ ]:
def binary_metrics_only(yv, ybin):
    """Operating-point metrics only (used inside the bootstrap; AUC/AUPRC skipped for speed)."""
    yv = np.asarray(yv); ybin = np.asarray(ybin)
    tp = int(((ybin == 1) & (yv == 1)).sum()); fn = int(((ybin == 0) & (yv == 1)).sum())
    tn = int(((ybin == 0) & (yv == 0)).sum()); fp = int(((ybin == 1) & (yv == 0)).sum())
    return {'sensitivity': tp / (tp + fn) if (tp + fn) else 0.0,
            'specificity': tn / (tn + fp) if (tn + fp) else 0.0,
            'ppv': tp / (tp + fp) if (tp + fp) else 0.0,
            'npv': tn / (tn + fn) if (tn + fn) else 0.0,
            'f1': 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0}


def patient_bootstrap_ci_bin(yv, p_pat, labels, B=N_BOOT, seed=SEED):
    """Patient-level bootstrap: AUC/AUPRC/Brier from the pooled probabilities;
    Sen/Spec/PPV/NPV/F1 recomputed per repeat within each resample, then averaged."""
    rng = np.random.default_rng(seed)
    n = len(yv)
    BIN = ['sensitivity', 'specificity', 'ppv', 'npv', 'f1']
    acc = {k: [] for k in ['auc', 'auprc', 'brier'] + BIN}
    for _ in range(B):
        ii = rng.integers(0, n, n)
        yb = yv[ii]
        if len(np.unique(yb)) < 2:
            continue
        acc['auc'].append(roc_auc_score(yb, p_pat[ii]))
        acc['auprc'].append(average_precision_score(yb, p_pat[ii]))
        acc['brier'].append(brier_score_loss(yb, p_pat[ii]))
        vals = {k: [] for k in BIN}
        for r in range(labels.shape[0]):
            mb = binary_metrics_only(yb, labels[r][ii])
            for k in BIN:
                vals[k].append(mb[k])
        for k in BIN:
            acc[k].append(float(np.mean(vals[k])))
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
            for k, v in acc.items() if v}


def resummarize_oof(yv, probs, splits, cuts=None, B=N_BOOT, seed=SEED):
    """Summary under the pooled-OOF convention (= **ensemble of repeated CV fits**):

    - pooled probability: each patient's 20 OOF predictions are averaged, i.e. the 20 repeated
        fits are treated as one ensemble; AUC/AUPRC/Brier are computed from it.
    - operating points: each fold's training threshold cuts[fi] classifies the test fold,
        summarised per repeat and averaged over the 20 repeats.
    - 95% CI: patient-level bootstrap; binary metrics recomputed per repeat in each resample.
    """
    n = len(yv)
    keys = ['auc', 'auprc', 'sensitivity', 'specificity', 'ppv', 'npv', 'f1', 'brier']
    BIN = ['sensitivity', 'specificity', 'ppv', 'npv', 'f1']
    per_patient = np.zeros((N_REPEATS, n))
    lab = np.zeros((N_REPEATS, n), dtype=int)
    rep = {k: [] for k in keys}
    for r in range(N_REPEATS):
        yy, pp, lb = [], [], []
        for f in range(N_SPLITS):
            fi = r * N_SPLITS + f
            te = splits[fi][1]
            cut = cuts[fi] if cuts is not None else youden_cutoff(yv[te], probs[fi])
            pr = np.asarray(probs[fi])
            yy.append(yv[te]); pp.append(pr)
            b = (pr >= cut).astype(int)
            lb.append(b)
            per_patient[r][te] = pr
            lab[r][te] = b
        yy = np.concatenate(yy); pp = np.concatenate(pp); lb = np.concatenate(lb)
        m = metrics_from_binary(yy, pp, lb)
        for k in keys:
            rep[k].append(m[k])
    p_pat = per_patient.mean(axis=0)
    pooled = {'auc': float(roc_auc_score(yv, p_pat)),
              'auprc': float(average_precision_score(yv, p_pat)),
              'brier': float(brier_score_loss(yv, p_pat))}
    for k in BIN:
        pooled[k] = float(np.mean(rep[k]))
    cut_mean = (float(np.mean([c for c in cuts if c is not None]))
                if cuts is not None else float(youden_cutoff(yv, p_pat)))
    return {'repeat_metrics': rep, 'p_patient': p_pat, 'labels': lab,
            'cutoff_pooled': cut_mean,
            'boot_ci': patient_bootstrap_ci_bin(yv, p_pat, lab, B=B, seed=seed),
            'pooled_point': pooled}

### `compute_nri_idi()`

In [ ]:
def compute_nri_idi(p_base, p_new, labels):
    event, nonevent = labels == 1, labels == 0
    up_e = np.sum(p_new[event] > p_base[event]); dn_e = np.sum(p_new[event] < p_base[event])
    nri_e = (up_e - dn_e) / max(1, event.sum())
    up_n = np.sum(p_new[nonevent] > p_base[nonevent]); dn_n = np.sum(p_new[nonevent] < p_base[nonevent])
    nri_ne = (up_n - dn_n) / max(1, nonevent.sum())
    idi = ((p_new[event].mean() if event.sum() else 0) - (p_base[event].mean() if event.sum() else 0)) - \
          ((p_new[nonevent].mean() if nonevent.sum() else 0) - (p_base[nonevent].mean() if nonevent.sum() else 0))
    return nri_e - nri_ne, idi

### `corrected_resampled_t()` - Nadeau-Bengio corrected repeated-resampling paired t-test.

In [ ]:
def corrected_resampled_t(diffs, n_train, n_test):
    """Nadeau-Bengio corrected repeated-resampling paired t-test.
    diffs: per-repeat differences (length J). Returns mean, lo, hi, p."""
    d = np.asarray(diffs, dtype=float)
    J = len(d)
    m = d.mean()
    var_corr = (1.0 / J + n_test / n_train) * d.var(ddof=1)
    se = np.sqrt(var_corr)
    if se == 0:
        return m, m, m, np.nan
    tcrit = t_dist.ppf(0.975, J - 1)
    tstat = m / se
    p = 2 * t_dist.sf(abs(tstat), J - 1)
    return m, m - tcrit * se, m + tcrit * se, p

### `patient_bootstrap_ci()` - patient-level bootstrap of pooled OOF predictions, percentile 95% CI.

In [ ]:
def patient_bootstrap_ci(yv, p, cutoff, B=N_BOOT, seed=SEED):
    """Patient-level bootstrap of pooled OOF predictions; percentile 95% CI."""
    rng = np.random.default_rng(seed)
    keys = ['auc', 'auprc', 'sensitivity', 'specificity', 'ppv', 'npv', 'f1', 'brier']
    acc = {k: [] for k in keys}
    n = len(yv)
    for _ in range(B):
        ii = rng.integers(0, n, n)
        yb, pb = yv[ii], p[ii]
        if len(np.unique(yb)) < 2:
            continue
        m = metrics_from_probs(yb, pb, cutoff)
        for k in keys:
            acc[k].append(m[k])
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in acc.items()}

## 4. Outer nested CV workflow

### `run_nested_cv()`

In [ ]:
RFECV_CURVES = []        # RFECV curve of each outer fold (Figure 3B)


def run_nested_cv():
    rskf_out = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS,
                                       random_state=SEED)
    splits = list(rskf_out.split(X_raw_all.values, y))
    n_train = int(len(y) * (N_SPLITS - 1) / N_SPLITS)
    n_test = len(y) - n_train

    FIXED = {                                   # fixed feature sets (inner tuning only)
        'Clinical_Only': clinical_cols,
        'Single_Immature': [biomarker_col],
        'Clinical_Immature': ALL8,
    }
    store = {k: {'probs': [None] * len(splits), 'cuts': [None] * len(splits)}
             for k in list(FIXED)}
    store['Parsimonious_Pipeline'] = {'probs': [None] * len(splits),
                                      'cuts': [None] * len(splits)}
    selection_log = []
    RFECV_CURVES.clear()

    t0 = time.time()
    for fi, (tr, te) in enumerate(splits):
        Xtr_df, Xte_df = X_raw_all.iloc[tr], X_raw_all.iloc[te]
        ytr = y[tr]
        # --- fixed models ---
        for k, feats in FIXED.items():
            b = fit_pipeline_on_train(Xtr_df, ytr, feats, seed=fi)
            store[k]['probs'][fi] = predict_bundle(b, Xte_df)
            store[k]['cuts'][fi] = b['cutoff']
        # --- simplified pipeline: RFE selects FINAL_K features inside each outer training fold ---
        best_feats, best_k, curve = rfecv_selection(Xtr_df, ytr, seed=fi)
        RFECV_CURVES.append({'outer_fold': fi, 'best_k': best_k, 'curve': curve})
        selection_log.append({'outer_fold': fi, 'n_selected': len(best_feats),
                              'best_k': best_k, 'selected': '|'.join(best_feats)})
        b = fit_pipeline_on_train(Xtr_df, ytr, best_feats, seed=fi)
        store['Parsimonious_Pipeline']['probs'][fi] = predict_bundle(b, Xte_df)
        store['Parsimonious_Pipeline']['cuts'][fi] = b['cutoff']
        if (fi + 1) % 10 == 0:
            el = time.time() - t0
            print(f'  outer {fi+1}/{len(splits)}  ({el:.0f}s, ETA {el/(fi+1)*(len(splits)-fi-1):.0f}s)',
                  flush=True)

    return store, splits, selection_log, (n_train, n_test)

### `summarize_nested()` - summary with the pooled-OOF convention (see the `resummarize_oof` docstring).

In [ ]:
def summarize_nested(store, splits, n_sizes):
    """Summary: pooled OOF = ensemble-of-repeated-CV convention (see resummarize_oof)."""
    out = {}
    for k, v in store.items():
        out[k] = resummarize_oof(y, v['probs'], splits, cuts=v['cuts'])
    return out

## 5. Run (~18 min)

In [ ]:
if __name__ == '__main__':
    print('=' * 80)
    print('Nested CV main loop (outer 5x20; inner: RFECV selection -> fit -> Youden threshold)')
    print('=' * 80)
    store, splits, selection_log, n_sizes = run_nested_cv()

    # ---------- feature-selection stability ----------
    sel_df = pd.DataFrame(selection_log)
    sel_df.to_csv(os.path.join(OUT_T, 'nested_selection_log.csv'), index=False)
    freq = {}
    for f in ALL8:
        freq[f] = np.mean([f in s.split('|') for s in sel_df['selected']])
    freq_s = pd.Series(freq).sort_values(ascending=False)
    print('\nSelection frequency across the 100 outer folds:')
    for f, v in freq_s.items():
        print(f'  {DISPLAY[f]}: {v:.2f}')
    print('  Distribution of subset size:', sel_df['n_selected'].value_counts().sort_index().to_dict())

    # ---------- second pass: fixed FINAL_K model on the same outer folds ----------
    # The 7- and 8-predictor models fix their feature set first and then run the 100 folds;
    # the simplified set is defined by selection frequency, so it is re-run to be comparable.
    FIXED_FEATS = sorted(freq_s.index, key=lambda f: (-freq_s[f], ALL8.index(f)))[:FINAL_K]
    print('\nFixed %d-predictor model (by selection frequency): %s'
          % (FINAL_K, ', '.join(DISPLAY[f] for f in FIXED_FEATS)))
    probs_f, cuts_f = [], []
    for fi, (tr, te) in enumerate(splits):
        b = fit_pipeline_on_train(X_raw_all.iloc[tr], y[tr], FIXED_FEATS, seed=fi)
        probs_f.append(predict_bundle(b, X_raw_all.iloc[te]))
        cuts_f.append(b['cutoff'])
    store['Parsimonious_Strategy'] = store.pop('Parsimonious_Pipeline')   # re-selected every fold; kept for the record
    store['Parsimonious_Pipeline'] = {'probs': probs_f, 'cuts': cuts_f}  # fixed four predictors

    summ = summarize_nested(store, splits, n_sizes)
    strat = summ['Parsimonious_Strategy']['pooled_point']
    pd.DataFrame([{'metric': m, 'value': strat[m]} for m in strat]).to_csv(
        os.path.join(OUT_T, 'strategy_estimate.csv'), index=False)
    print('  strategy-level (re-select each fold) pooled AUROC = %.4f; fixed %d predictors = %.4f'
          % (strat['auc'], FINAL_K, summ['Parsimonious_Pipeline']['pooled_point']['auc']))

    # ---------- print the four-model results ----------
    for k in ['Clinical_Only', 'Single_Immature', 'Clinical_Immature', 'Parsimonious_Pipeline']:
        s = summ[k]
        aucs = s['repeat_metrics']['auc']
        lo, hi = s['boot_ci']['auc']
        print(f"\n{k}: repeat-mean AUROC = {np.mean(aucs):.4f}; "
              f"pooled AUROC = {s['pooled_point']['auc']:.4f} "
              f"(patient-bootstrap 95% CI {lo:.3f}-{hi:.3f})")

    with open(os.path.join(OUT_T, 'nested_cv_results.pkl'), 'wb') as f:
        pickle.dump({'store_probs': {k: v['probs'] for k, v in store.items()},
                     'store_cuts': {k: v['cuts'] for k, v in store.items()},
                     'selector': SELECTOR,
                     'splits': splits, 'y': y, 'codes': codes,
                     'summary': summ, 'selection_log': selection_log,
                     'n_sizes': n_sizes}, f)
    print('\nSaved nested_cv_results.pkl')

---

# Part 2 - full-data derivation / four algorithms / single markers / subgroup / table data

Reads `nested_cv_results.pkl` written by Part 1; the outer CV is not re-run.

## Read the Part 1 results

In [ ]:
PKL = os.path.join(OUT_T, 'nested_cv_results.pkl')
with open(PKL, 'rb') as f:
    R = pickle.load(f)
splits, summ, n_sizes = R['splits'], R['summary'], R['n_sizes']
n_train, n_test = n_sizes

MODEL_KEYS = ['Clinical_Only', 'Single_Immature', 'Clinical_Immature', 'Parsimonious_Pipeline']
MODEL_HEAD = {
    'Clinical_Only': 'Clinical model (7 predictors)',
    'Single_Immature': 'CD10\u207b neutrophils alone',
    'Clinical_Immature': 'Combined model (8 predictors)',
    'Parsimonious_Pipeline': 'Simplified model',
}

## A. Full-data derivation (Figure 3)

## B. sTable S3 / S4 - four models and paired comparisons

### `model_tables()`

In [ ]:
def _holm_adjust(pvals):
    """Step-down Holm-Bonferroni correction (family = all supplied p values)."""
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    prev = 0.0
    for rank, idx in enumerate(order):
        val = min(1.0, (m - rank) * p[idx])
        val = max(val, prev)
        adj[idx] = val
        prev = val
    return adj


def _bh_adjust(pvals):
    """Benjamini-Hochberg FDR (q values); assumes independence or PRDS."""
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    q = np.empty(m)
    prev = 1.0
    for rank in range(m - 1, -1, -1):
        idx = order[rank]
        val = min(prev, p[idx] * m / (rank + 1))
        q[idx] = val
        prev = val
    return q


def model_tables():
    """sTable S3 / S4: common pooled (ensemble-of-repeated-CV) convention.

    - point estimates: metrics on the pooled probabilities (same source as Figure 4)
    - dAUROC / dAUPRC / dBrier / NRI / IDI: pooled paired patient bootstrap
    - multiplicity: 3 comparisons x 5 metrics = 15 tests; raw P and Holm-adjusted P
    - Nadeau-Bengio (repeat level) is a sensitivity analysis only -> separate CSV
    """
    print('\n[E] sTable S3/S4 summary (pooled convention + Holm correction)...', flush=True)
    keys = MODEL_KEYS
    desc = {}
    for k in keys:
        s = summ[k]
        desc[k] = {m: (float(s['pooled_point'][m]), s['boot_ci'][m][0], s['boot_ci'][m][1])
                   for m in ['auc', 'auprc', 'sensitivity', 'specificity', 'ppv', 'npv', 'f1']}

    rng_p = np.random.default_rng(SEED)
    n = len(y)

    def _metric(yy, pp, metric):
        if metric == 'auc':
            return roc_auc_score(yy, pp)
        if metric == 'auprc':
            return average_precision_score(yy, pp)
        return brier_score_loss(yy, pp)

    def _ci_p(pt, draws):
        draws = np.asarray(draws, float)
        lo, hi = np.percentile(draws, [2.5, 97.5])
        p = 2 * min((draws <= 0).mean(), (draws >= 0).mean())
        return float(pt), float(lo), float(hi), float(min(1.0, max(p, 2.0 / max(len(draws), 1))))

    def boot_delta(k_new, k_ref, metric, B=N_BOOT):
        p1 = np.asarray(summ[k_new]['p_patient'], float)
        p0 = np.asarray(summ[k_ref]['p_patient'], float)
        pt = _metric(y, p1, metric) - _metric(y, p0, metric)
        ds = []
        for _ in range(B):
            ii = rng_p.integers(0, n, n)
            yb = y[ii]
            if len(np.unique(yb)) < 2:
                continue
            ds.append(_metric(yb, p1[ii], metric) - _metric(yb, p0[ii], metric))
        return _ci_p(pt, ds)

    def boot_nri_idi(k_new, k_ref, B=N_BOOT):
        p1 = np.asarray(summ[k_new]['p_patient'], float)
        p0 = np.asarray(summ[k_ref]['p_patient'], float)
        nri0, idi0 = compute_nri_idi(p0, p1, y)
        nris, idis = [], []
        for _ in range(B):
            ii = rng_p.integers(0, n, n)
            yb = y[ii]
            if len(np.unique(yb)) < 2:
                continue
            a, b = compute_nri_idi(p0[ii], p1[ii], yb)
            nris.append(a); idis.append(b)
        return _ci_p(nri0, nris), _ci_p(idi0, idis)

    def rep_diffs(k_new, k_ref, metric):
        return (np.array(summ[k_new]['repeat_metrics'][metric])
                - np.array(summ[k_ref]['repeat_metrics'][metric]))

    pairs = [('Combined model (8 predictors) vs Clinical model (7 predictors)',
              'Clinical_Immature', 'Clinical_Only'),
             ('Simplified model vs Clinical model (7 predictors)',
              'Parsimonious_Pipeline', 'Clinical_Only'),
             ('Simplified model vs Combined model (8 predictors)',
              'Parsimonious_Pipeline', 'Clinical_Immature')]
    cmp_rows, nb_rows, raw_p = [], [], []
    for label, kn, kr in pairs:
        row = {'comparison': label}
        for metric, tag in [('auc', 'd_auc'), ('auprc', 'd_auprc'), ('brier', 'd_brier')]:
            m_, lo, hi, p = boot_delta(kn, kr, metric)
            row[tag] = (m_, lo, hi); row['p_' + tag] = p
            m2, lo2, hi2, p2 = corrected_resampled_t(rep_diffs(kn, kr, metric), n_train, n_test)
            row['nb_' + tag] = (m2, lo2, hi2); row['nb_p_' + tag] = p2
            nb_rows.append({'comparison': label, 'metric': metric,
                            'delta_nb': m2, 'nb_lo': lo2, 'nb_hi': hi2, 'nb_p': p2})
        nri_res, idi_res = boot_nri_idi(kn, kr)   # a single bootstrap run
        row['nri'], row['p_nri'] = nri_res[:3], nri_res[3]
        row['idi'], row['p_idi'] = idi_res[:3], idi_res[3]
        cmp_rows.append(row)
        raw_p += [row['p_d_auc'], row['p_d_auprc'], row['p_d_brier'], row['p_nri'], row['p_idi']]

    # Holm correction (family = 15 tests)
    adj = _holm_adjust(raw_p)
    fdr = _bh_adjust(raw_p)
    for i, row in enumerate(cmp_rows):
        for j, tag in enumerate(['d_auc', 'd_auprc', 'd_brier', 'nri', 'idi']):
            row['p_holm_' + tag] = float(adj[i * 5 + j])
            row['q_bh_' + tag] = float(fdr[i * 5 + j])
        print('  %-52s dAUC %+.4f  P=%.4f (Holm %.4f)'
              % (row['comparison'][:52], row['d_auc'][0], row['p_d_auc'],
                 row['p_holm_d_auc']), flush=True)

    pd.DataFrame(nb_rows).to_csv(os.path.join(OUT_T, 'table_s5_nadeau_bengio.csv'), index=False)

    # non-inferiority (lower 95% CI bound of the pooled delta)
    ni_rows = []
    for r in cmp_rows[1:]:
        ni_rows.append({'comparison': r['comparison'], 'delta_auc_mean': r['d_auc'][0],
                        'delta_auc_lo': r['d_auc'][1], 'delta_auc_hi': r['d_auc'][2],
                        'margin': MARGIN,
                        'conclusion': 'Passed' if r['d_auc'][1] > MARGIN else 'Not passed'})
    pd.DataFrame(ni_rows).to_csv(os.path.join(OUT_T, 'non_inferiority_3models.csv'), index=False)
    return desc, cmp_rows

## C. Single markers (sTable S5)

### `individual_markers()`

In [ ]:
def individual_markers():
    """Single predictor: the same pipeline as S4 (impute -> standardise -> regularised LR),
    with one feature; per-fold fitting, pooled OOF and patient bootstrap, matching the
    'CD10- neutrophils alone' column of S4 rather than a raw-value ROC."""
    print('\n[C] Single predictors (pipeline convention)...', flush=True)
    d1s = pd.read_csv(os.path.join(ROOT, 'data', 'NE_baseline_D1.csv'), encoding='utf-8-sig')
    d1s = d1s[d1s['Code'].isin(codes)].sort_values('Code').reset_index(drop=True)
    cols = {'INR': 'INR', 'Platelet count': 'PLT',
            'CD10- neutrophils, %': biomarker_col, 'Day 1 SIC score': 'SIC'}
    out = {}
    rng = np.random.default_rng(SEED)
    for name, col in cols.items():
        Xm = d1s[[col]].rename(columns={col: name})
        probs = [None] * len(splits)
        for fi, (tr, te) in enumerate(splits):
            b = fit_pipeline_on_train(Xm.iloc[tr], y[tr], [name], seed=fi)
            probs[fi] = predict_bundle(b, Xm.iloc[te])
        pp = np.zeros(len(y)); cnt = np.zeros(len(y))
        for fi, (tr, te) in enumerate(splits):
            pp[te] += probs[fi]; cnt[te] += 1
        p_pat = pp / np.maximum(cnt, 1)
        auc_pt = float(roc_auc_score(y, p_pat))
        ap_pt = float(average_precision_score(y, p_pat))
        ba, bp = [], []
        for _ in range(N_BOOT):
            ii = rng.integers(0, len(y), len(y))
            if len(np.unique(y[ii])) < 2:
                continue
            ba.append(roc_auc_score(y[ii], p_pat[ii]))
            bp.append(average_precision_score(y[ii], p_pat[ii]))
        out[name] = {'auc': (auc_pt, *map(float, np.percentile(ba, [2.5, 97.5]))),
                     'auprc': (ap_pt, *map(float, np.percentile(bp, [2.5, 97.5]))),
                     'n': int(np.isfinite(d1s[col]).sum())}
        print(f"  {name}: AUROC {out[name]['auc'][0]:.3f} "
              f"({out[name]['auc'][1]:.3f}-{out[name]['auc'][2]:.3f})", flush=True)
    return out

## D. Four algorithms under nested CV (sTable S6 / Figure S6; input = ALL8)

### `run_algorithms()`

In [ ]:
def run_algorithms():
    print('\n[D] Four algorithms under nested CV (input = ALL8, 8 predictors)...', flush=True)
    XC = X_raw_all[ALL8].copy()      # all 8 predictors (7 clinical + CD10-)
    algos = {
        'Logistic Regression': dict(
            make=lambda: LogisticRegression(penalty='elasticnet', solver='saga',
                                            max_iter=LR_MAX_ITER, tol=LR_TOL,
                                            random_state=SEED),
            params=param_dist, scaled=True, n_iter=20),
        'Gradient Boosting': dict(
            make=lambda: GradientBoostingClassifier(random_state=SEED),
            params={'n_estimators': randint(50, 200), 'learning_rate': uniform(0.01, 0.3),
                    'max_depth': randint(3, 5), 'min_samples_split': randint(2, 20)},
            scaled=False, n_iter=20),
        'SVM': dict(
            make=lambda: SVC(probability=True, random_state=SEED),
            params={'C': uniform(0.1, 100), 'kernel': ['rbf', 'linear'],
                    'gamma': ['scale', 'auto']},
            scaled=True, n_iter=20),
        'Gaussian Naive Bayes': dict(make=lambda: GaussianNB(), params={}, scaled=True,
                                     n_iter=0),
    }
    res = {}
    t0 = time.time()
    for name, cfg in algos.items():
        probs = [None] * len(splits)
        cuts = [None] * len(splits)
        for fi, (tr, te) in enumerate(splits):
            imp = IterativeImputer(max_iter=1000, random_state=0).fit(XC.iloc[tr])
            Xtr = imp.transform(XC.iloc[tr]); Xte = imp.transform(XC.iloc[te])
            if cfg['scaled']:
                sc = StandardScaler().fit(Xtr)
                Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
            if cfg['n_iter'] > 0:
                s = RandomizedSearchCV(cfg['make'](), cfg['params'], n_iter=cfg['n_iter'],
                                       cv=5, scoring='roc_auc', random_state=fi, n_jobs=N_JOBS)
                s.fit(Xtr, y[tr]); m = s.best_estimator_
            else:
                m = cfg['make'](); m.fit(Xtr, y[tr])
            probs[fi] = m.predict_proba(Xte)[:, 1]
            cuts[fi] = youden_cutoff(y[tr], m.predict_proba(Xtr)[:, 1])
            if (fi + 1) % 25 == 0:
                print(f'    {name}: {fi+1}/{len(splits)} ({time.time()-t0:.0f}s)', flush=True)
        # pooled OOF convention (same as summarize_nested):
        # the 5 folds are combined per repeat; the Youden threshold comes from those pooled predictions;
        # 20 repeats averaged per patient -> point estimate; patient-level bootstrap 95% CI.
        summ = resummarize_oof(y, probs, splits, cuts=cuts)
        res[name] = {'repeat_metrics': summ['repeat_metrics'],
                     'p_patient': summ['p_patient'],
                     'cutoff': summ['cutoff_pooled'],
                     'per_patient': None,
                     'boot_ci': summ['boot_ci'],
                     'pooled_point': summ['pooled_point'],
                     'store_probs': probs, 'store_cuts': cuts}
        per_patient = np.zeros((N_REPEATS, len(y)))
        for r in range(N_REPEATS):
            for f in range(N_SPLITS):
                fi = r * N_SPLITS + f
                per_patient[r][splits[fi][1]] = probs[fi]
        res[name]['per_patient'] = per_patient
        print(f'  {name}: pooled AUROC = {res[name]["pooled_point"]["auc"]:.4f} '
              f'({res[name]["boot_ci"]["auc"][0]:.3f}-{res[name]["boot_ci"]["auc"][1]:.3f})',
              flush=True)
    return res

## E. NSIC subgroup (sTable S7 / Figure S7)

### `nsic_subgroup()`

In [ ]:
def nsic_subgroup(pars_features):
    """NSIC subgroup: **descriptive only** (n, events, distribution of predicted risk); no performance metric.

    43 patients and 5 events: an AUC would be dominated by variance, hence descriptive only.
    """
    print('\n[E] NSIC subgroup (descriptive only)...', flush=True)
    d1s = pd.read_csv(os.path.join(ROOT, 'data', 'NE_baseline_D1.csv'), encoding='utf-8-sig')
    d1s = d1s[d1s['Code'].isin(codes)].sort_values('Code').reset_index(drop=True)
    sub = (d1s['SIC_D1'] == 'NSIC').values
    ys = y[sub]
    out = {'n': int(sub.sum()), 'events': int(ys.sum()), 'models': {},
           'pars_features': list(pars_features)}
    pairs = [('Clinical model (7 predictors)', 'Clinical_Only'),
             ('Combined model (8 predictors)', 'Clinical_Immature'),
             ('Simplified model (%d predictors)' % len(pars_features), 'Parsimonious_Pipeline')]
    for label, key in pairs:
        p = np.asarray(R['summary'][key]['p_patient'])[sub]
        ev, ne = p[ys == 1], p[ys == 0]
        out['models'][label] = {
            'n_event': int(len(ev)), 'n_nonevent': int(len(ne)),
            'median_event': float(np.median(ev)) if len(ev) else float('nan'),
            'iqr_event': (float(np.percentile(ev, 25)), float(np.percentile(ev, 75))) if len(ev) else (float('nan'),) * 2,
            'median_nonevent': float(np.median(ne)) if len(ne) else float('nan'),
            'iqr_nonevent': (float(np.percentile(ne, 25)), float(np.percentile(ne, 75))) if len(ne) else (float('nan'),) * 2,
        }
        print('  %s: event median %.3f / non-event median %.3f'
              % (label, out['models'][label]['median_event'],
                 out['models'][label]['median_nonevent']), flush=True)
    return out

## F. Feature-selection process (for Figure 3; no separate table)

In [ ]:
def final_feature_set():
    """Final simplified model: top FINAL_K predictors by **selection frequency** (ties by |coef|).

    Frequencies come from the 100 outer folds of Part 1;
    coefficients come from the models actually fitted there (Figure 3A).
    Note: unbiased performance still comes from the RFE pipeline (re-selection per fold);
    this function only fixes the model that is finally delivered.
    """
    log = R['selection_log']
    n = len(log)
    freq = {f: 0 for f in ALL8}
    for rec in log:
        for f in str(rec.get('selected', '')).split('|'):
            if f in freq:
                freq[f] += 1
    order = sorted(ALL8, key=lambda f: (-freq[f], f))
    feats = order[:FINAL_K]
    print('\n[H] Final simplified model (top %d by selection frequency):' % FINAL_K, flush=True)
    for f in order:
        mark = ' <=' if f in feats else ''
        print('   %-30s %3d/%d  %5.1f%%%s' % (DISPLAY[f], freq[f], n, 100 * freq[f] / n, mark),
              flush=True)
    return feats

In [ ]:
# Selection stability is no longer tabulated (the former sTable S8 was removed);
# percentages in Figure 3A come directly from outer_fold_importance().

In [ ]:
def outer_fold_importance():
    """Data source for Figure 3A: coefficients of the parsimonious model **inside each outer fold**.

    For every outer fold of Part 1, that fold's feature set is re-fitted with the same seed
    (identical to run_nested_cv) and coefficients recorded; **unselected folds contribute 0**,
    so a bar is the mean contribution over 100 folds (effect size x selection frequency);
    the right-hand percentages report stability separately.
    """
    print('\n[G] Outer-fold coefficients (Figure 3A source)...', flush=True)
    log = R['selection_log']
    rows, sel_flags = [], []
    for fi, (tr, te) in enumerate(R['splits']):
        feats = [f for f in str(log[fi].get('selected', '')).split('|') if f]
        b = fit_pipeline_on_train(X_raw_all.iloc[tr], y[tr], feats, seed=fi)
        coef = dict(zip(feats, b['model'].coef_[0]))
        rows.append({f: coef.get(f, 0.0) for f in ALL8})
        sel_flags.append({f: (f in feats) for f in ALL8})
    recs = []
    for f in ALL8:
        v_all = np.array([rows[i][f] for i in range(len(rows))], float)
        n_sel = int(sum(sel_flags[i][f] for i in range(len(rows))))
        recs.append({'feature': f, 'display': DISPLAY[f],
                     'mean_coef': float(v_all.mean()),
                     'std_coef': float(v_all.std(ddof=1)),
                     'n_selected': n_sel,
                     'selection_pct': 100.0 * n_sel / len(rows)})
    out = pd.DataFrame(recs)
    out['abs_coef'] = out['mean_coef'].abs()
    out = out.sort_values('abs_coef', ascending=False).reset_index(drop=True)
    out.to_csv(os.path.join(OUT_T, 'outer_fold_coefficients.csv'), index=False)
    print(out[['display', 'mean_coef', 'n_selected', 'selection_pct']].to_string(index=False),
          flush=True)
    return out

## Run (~20 min)

In [ ]:
if __name__ == '__main__':
    pars_features = final_feature_set()      # Figure 3 / parsimonious model definition
    desc, cmp_rows = model_tables()          # sTable S3-S4: four models + paired comparisons
    markers = individual_markers()           # sTable S5: single markers (same pipeline as S3)
    algo_res = run_algorithms()              # sTable S6 / Fig S6: four algorithms (input ALL8)
    subgroup = nsic_subgroup(pars_features)  # sTable S7: NSIC subgroup
    imp_outer = outer_fold_importance()      # Figure 3A: within-outer-fold coefficients
    with open(os.path.join(OUT_T, 'nested_cv_stage2.pkl'), 'wb') as f:
        pickle.dump({'pars_features': pars_features,
                     'algo_res': algo_res, 'markers': markers, 'subgroup': subgroup,
                     'desc': desc, 'cmp_rows': cmp_rows,
                     'imp_outer': imp_outer,
                     'rfecv_curves': RFECV_CURVES}, f)
    print('\nSaved nested_cv_stage2.pkl')

---

# Part 3 - figures (Figure 3/4, Supplementary Figures S6/S7)

In [ ]:
import os
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve

try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # Jupyter environment
    HERE = os.getcwd()
ROOT = HERE
for _ in range(5):
    if os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'outputs')):
        break
    ROOT = os.path.dirname(ROOT)
DATA = os.path.join(ROOT, 'data')
OUT_T = os.path.join(ROOT, 'outputs', 'tables')
OUT_F = os.path.join(ROOT, 'outputs', 'figures')
FIG = os.path.join(ROOT, 'figures')
os.makedirs(OUT_F, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

In [ ]:
d1 = pd.read_csv(os.path.join(DATA, 'NE_baseline_D1.csv'), encoding='utf-8-sig')
d3lab = pd.read_csv(os.path.join(DATA, 'NE_baseline_D3.csv'), encoding='utf-8-sig')
d3lab = d3lab[['Code', 'SIC']].dropna().set_index('Code')
d1 = d1[d1['Code'].isin(d3lab.index)].sort_values('Code').reset_index(drop=True)
y = (d3lab.loc[d1['Code'], 'SIC'].values == 1).astype(int)   # Day-3 SIC (score >= 4, binary)

DISPLAY = {
    'INR': 'INR', 'PLT': 'Platelet count', 'lac': 'Lactate',
    'SOFA D1 Circulation': 'SOFA cardiovascular subscore',
    'SOFA D1 Respiratory System': 'SOFA respiratory subscore',
    'VT_D1': 'Mechanical ventilation', 'SOFA_Creatinine': 'SOFA renal subscore',
    'immature_Neu_percentage': 'CD10- neutrophils,%',
}

In [ ]:
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']   # silent fallback, no findfont warnings
matplotlib.rcParams['axes.unicode_minus'] = False

with open(os.path.join(OUT_T, 'nested_cv_results.pkl'), 'rb') as f:
    R = pickle.load(f)
with open(os.path.join(OUT_T, 'nested_cv_stage2.pkl'), 'rb') as f:
    S2 = pickle.load(f)

summ = R['summary']
pars_features = S2['pars_features']
N_PARS = len(pars_features)
PARS_LABEL = f'Simplified model ({N_PARS} predictors)'

BLACK, BLUE, GREEN, RED = '#000000', '#1A4A8C', '#339C62', '#B2182B'
GRADIENT = ['#0D2B53', '#1A4A8C', '#2166AC', '#4393C3', '#92C5DE',
            '#F4A582', '#D6604D', '#B2182B', '#67001F']

### `save()`

In [ ]:
def save(fig, name, dpi=300, tight=True):
    """Save png/pdf/tif; tight=True crops to content (matching the reference figures)."""
    from PIL import Image
    png = os.path.join(FIG, f'{name}.png')
    kw = dict(dpi=dpi, facecolor='white')
    if tight:
        kw.update(bbox_inches='tight', pad_inches=0.02)
    fig.savefig(png, **kw)
    fig.savefig(os.path.join(FIG, f'{name}.pdf'), **kw)
    fig.savefig(os.path.join(OUT_F, f'{name}.png'), **kw)
    fig.savefig(os.path.join(OUT_F, f'{name}.pdf'), **kw)
    Image.open(png).convert('RGB').save(os.path.join(FIG, f'{name}.tif'), dpi=(300, 300))
    plt.close(fig)
    print(f'  saved {name}', flush=True)

### `figure3()`

In [ ]:
# ============================================================ model order / colours
MODEL_ORDER = ['Clinical_Only', 'Single_Immature', 'Clinical_Immature', 'Parsimonious_Pipeline']
F4_LABEL = {
    'Clinical_Only': 'Clinical model (7 predictors)',
    'Single_Immature': 'CD10$^{-}$ neutrophils (%)',
    'Clinical_Immature': 'Combined model (8 predictors)',
    'Parsimonious_Pipeline': PARS_LABEL,
}
F4_COLOR = {'Clinical_Only': BLACK, 'Single_Immature': GREEN,
            'Clinical_Immature': BLUE, 'Parsimonious_Pipeline': RED}

In [ ]:
def figure3():
    """Figure 3 - A: predictor stability and standardised coefficients; B: inner-CV discrimination by size.

    A: each fold is fitted with its own selected predictors, unselected ones contribute 0; mean ± SD
       are taken over the 100 folds; right-hand percentages are selection rates.
    B: each k summarises inner-CV AUROC across folds; the band is SD (descriptive, not a CI).
    """
    S = 1.18   # scale relative to 8.4 x 10.6 in (matched to the reference figure)

    imp = S2['imp_outer'].copy()
    imp = imp.sort_values(['selection_pct', 'abs_coef'], ascending=[False, False]).reset_index(drop=True)

    fig = plt.figure(figsize=(8.4 * S, 10.6 * S))
    ax1 = fig.add_axes([0.180, 0.590, 0.790, 0.335])   # A
    ax2 = fig.add_axes([0.180, 0.068, 0.790, 0.435])   # B

    # ---- A ----
    LAB = {'immature_Neu_percentage': 'CD10$^{-}$ neutrophils (%)',
           'SOFA D1 Circulation': 'SOFA cardiovascular',
           'SOFA D1 Respiratory System': 'SOFA respiratory',
           'SOFA_Creatinine': 'SOFA renal'}
    names = [LAB.get(f, d) for f, d in zip(imp['feature'], imp['display'])]
    vals = imp['mean_coef'].values
    errs = imp['std_coef'].values
    pct = imp['selection_pct'].values
    ypos = np.arange(len(names))

    # colour by selection frequency, hue fixed to the primary blue #315C97
    # frequency -> intensity: the scale is stretched so that 90/95/100 stay distinguishable
    _f = np.asarray(pct, float)
    _fx = [1, 3, 15, 23, 90, 95, 100]
    _ty = [0.12, 0.20, 0.34, 0.42, 0.72, 0.86, 1.00]
    _t = np.interp(_f, _fx, _ty)
    _freq_cmap = mcolors.LinearSegmentedColormap.from_list(
        'freq_blue', ['#EEF2F8', '#315C97'])
    colors = [mcolors.to_hex(_freq_cmap(x)) for x in _t]
    edges = [mcolors.to_hex(np.array(mcolors.to_rgb(c)) * 0.78) for c in colors]
    for i in range(len(names)):
        ax1.barh(ypos[i], vals[i], xerr=errs[i], height=0.62,
                 color=colors[i], edgecolor=edges[i], linewidth=0.6,
                 capsize=4, ecolor='#7A7A7A',
                 error_kw=dict(elinewidth=1.1, capthick=1.1))

    ax1.axvline(0, color='#333333', lw=1.0)
    xmax = float(np.max(vals + errs)); xmin = float(np.min(vals - errs))
    ax1.set_xlim(min(xmin, 0) - 0.05, xmax + 0.50)
    ax1.set_aspect('auto')
    ax1.set_ylim(len(names) - 0.45, -0.95)

    for i, p in enumerate(pct):
        ax1.text(1 - 0.006, i, '%.0f%%' % p, transform=ax1.get_yaxis_transform(),
                 va='center', ha='right', fontsize=12.5 * S, fontweight='bold',
                 color='#333333' if i < 4 else '#8A8F95')
    ax1.text(0.995, 0.985, 'Selection frequency', transform=ax1.transAxes,
             ha='right', va='top', fontsize=11.5 * S, fontweight='bold', color='#555555')

    ax1.set_yticks(ypos)
    ax1.set_yticklabels(names, fontsize=13 * S, fontweight='bold', color='#1A1A1A')
    ax1.set_xlabel('Mean standardized coefficient (log-odds)', fontsize=15 * S, fontweight='bold')
    ax1.tick_params(axis='x', labelsize=13 * S)
    ax1.grid(axis='x', alpha=0.25)

    from matplotlib.patches import Patch as _Patch
    from matplotlib.lines import Line2D as _Line2D
    ax1.legend(handles=[
        _Patch(facecolor='#8C8C8C', edgecolor='#333333', linewidth=0.6,
               label='Mean standardized coefficient \u00b1 SD')],
        loc='lower left', fontsize=10.5 * S, frameon=True, framealpha=0.95,
        borderpad=0.35, handlelength=1.4, bbox_to_anchor=(0.004, 0.012))

    fig.text(0.012, 0.932, 'A', fontsize=20 * S, fontweight='bold', va='bottom')

    # ---- B ----
    curves = S2.get('rfecv_curves', [])
    if not curves:
        ax2.text(0.5, 0.5, 'RFECV curve unavailable\n(run Part 1 + Part 2)',
                 ha='center', va='center', fontsize=12 * S, color='#888888')
        ax2.set_xticks([]); ax2.set_yticks([])
        save(fig, 'Figure 3')
        return

    ks = sorted({int(k) for c in curves for k in c['curve']}, reverse=True)
    mean = np.array([np.mean([c['curve'][k] for c in curves if k in c['curve']]) for k in ks])
    sd = np.array([np.std([c['curve'][k] for c in curves if k in c['curve']], ddof=1) for k in ks])

    ax2.axhline(mean[ks.index(max(ks))], color='#6E6E6E', lw=1.1, ls='--',
                label='Full model (%d predictors)' % max(ks))
    ax2.fill_between(ks, mean - sd, mean + sd, color='#315C97', alpha=0.12, lw=0,
                     label='\u00b1 1 SD across outer folds')
    ax2.plot(ks, mean, 'o-', color='#315C97', lw=2.0, ms=6.5 * S,
             label='Mean inner-CV AUROC')
    for kx, my in zip(ks, mean):
        if kx == min(ks):          # right-most point (inverted axis): label to the lower left, away from the line
            dx, dy, ha, va = -12, -14, 'right', 'top'
        elif kx == max(ks):        # left-most point: label to the upper left
            dx, dy, ha, va = 10, 12, 'left', 'bottom'
        else:
            dx, dy, ha, va = 0, 13, 'center', 'bottom'
        ax2.annotate('%.3f' % my, (kx, my), textcoords='offset points', xytext=(dx, dy),
                     ha=ha, va=va, fontsize=13.5 * S, color='#315C97', fontweight='bold')
    if FINAL_K in ks:
        ax2.plot([FINAL_K], [mean[ks.index(FINAL_K)]], marker='D', ls='', ms=9.5 * S,
                 markerfacecolor='#B92F40', markeredgecolor='white', markeredgewidth=1.5, zorder=6,
                 label='Selected model size (%d predictors)' % FINAL_K)

    ax2.set_xlabel('Number of predictors', fontsize=15 * S, fontweight='bold')
    ax2.set_ylabel('Inner-CV AUROC', fontsize=15 * S, fontweight='bold')
    ax2.set_xticks(ks)
    ax2.invert_xaxis()
    ax2.set_ylim(0.910, 0.950)
    ax2.tick_params(labelsize=13 * S)
    ax2.grid(alpha=0.25, ls=':')
    ax2.legend(loc='lower left', fontsize=10.5 * S, framealpha=0.9)
    fig.text(0.012, 0.492, 'B', fontsize=20 * S, fontweight='bold', va='bottom')

    save(fig, 'Figure 3')


def figure4():
    """Figure 4 - A ROC; B calibration; C decision curves."""
    fig = plt.figure(figsize=(12.4, 3.35))
    axA = fig.add_axes([0.050, 0.175, 0.262, 0.775])
    axB = fig.add_axes([0.378, 0.175, 0.262, 0.775])
    axC = fig.add_axes([0.706, 0.175, 0.262, 0.775])

    # ---- A: ROC ----
    axA.plot([0, 1], [0, 1], color='#999999', lw=1.3, ls='--')
    for k in MODEL_ORDER:
        s = summ[k]
        fpr, tpr, _ = roc_curve(y, s['p_patient'])
        auc_p = s['pooled_point']['auc']
        lo, hi = s['boot_ci']['auc']
        axA.plot(fpr, tpr, color=F4_COLOR[k], lw=2.2,
                 label=f'{F4_LABEL[k].split(" (")[0]}: {auc_p:.3f} ({lo:.3f}\u2013{hi:.3f})')
    axA.set_xlim(0, 1); axA.set_ylim(0, 1.02)
    axA.set_xlabel('1 - Specificity', fontsize=12, fontweight='bold')
    axA.set_ylabel('Sensitivity', fontsize=12, fontweight='bold')
    axA.legend(loc='lower right', fontsize=7.6, framealpha=0.88)
    axA.tick_params(labelsize=10)
    axA.grid(alpha=0.3, ls=':')
    axA.text(-0.18, 1.03, 'A', transform=axA.transAxes, fontsize=17, fontweight='bold')

    # ---- B: calibration (one record per patient: pooled OOF probability) ----
    def _wilson(k_, n_):
        if n_ == 0:
            return 0.0, 1.0
        z = 1.96
        ph = k_ / n_
        den = 1 + z ** 2 / n_
        centre = (ph + z ** 2 / (2 * n_)) / den
        half = z * np.sqrt(ph * (1 - ph) / n_ + z ** 2 / (4 * n_ ** 2)) / den
        return max(0.0, centre - half), min(1.0, centre + half)

    axB.plot([0, 1], [0, 1], color='#999999', lw=1.2, ls='--')
    calib_rows = []
    for k in MODEL_ORDER:
        p = np.asarray(summ[k]['p_patient'], float)
        qs = np.quantile(p, np.linspace(0, 1, 6))
        qs[0] -= 1e-9; qs[-1] += 1e-9
        mx, ob, lo, hi = [], [], [], []
        for i in range(5):
            m = (p > qs[i]) & (p <= qs[i + 1])
            if m.sum() < 5:
                continue
            mx.append(p[m].mean()); ob.append(y[m].mean())
            a, b = _wilson(int(y[m].sum()), int(m.sum()))
            lo.append(a); hi.append(b)
        axB.errorbar(mx, ob, yerr=[np.array(ob) - np.array(lo), np.array(hi) - np.array(ob)],
                     fmt='o-', color=F4_COLOR[k], lw=1.4, ms=5, capsize=3, elinewidth=1.1,
                     label=F4_LABEL[k].split(' (')[0])
        # calibration slope / intercept (logit scale, one record per patient)
        eps = 1e-6
        pc = np.clip(p, eps, 1 - eps)
        lp = np.log(pc / (1 - pc))
        import statsmodels.api as _sm
        _fit = _sm.Logit(y, _sm.add_constant(lp.reshape(-1, 1))).fit(disp=0)
        calib_rows.append({'model': k, 'slope': float(_fit.params[1]),
                           'intercept': float(_fit.params[0])})
    pd.DataFrame(calib_rows).to_csv(
        os.path.join(OUT_T, 'calibration_slope_intercept.csv'), index=False, encoding='utf-8-sig')
    axB.set_xlim(0, 1.0); axB.set_ylim(0, 1.0)
    axB.set_xlabel('Predicted probability', fontsize=12, fontweight='bold')
    axB.set_ylabel('Observed proportion', fontsize=12, fontweight='bold')
    axB.legend(loc='lower right', fontsize=7.6, framealpha=0.88)
    axB.tick_params(labelsize=10)
    axB.grid(alpha=0.3, ls=':')
    axB.text(-0.18, 1.03, 'B', transform=axB.transAxes, fontsize=17, fontweight='bold')

    # ---- C: DCA ----
    thresholds = np.linspace(0.01, 0.99, 99)
    prev = y.mean()
    nb_all = prev - (1 - prev) * thresholds / (1 - thresholds)

    def net_benefit(yv, p, pt):
        n = len(yv)
        pred = p >= pt
        tp = np.sum(pred & (yv == 1)); fp = np.sum(pred & (yv == 0))
        return tp / n - fp / n * pt / (1 - pt)

    for k in MODEL_ORDER:
        p = summ[k]['p_patient']
        nb = np.array([net_benefit(y, p, pt) for pt in thresholds])
        axC.plot(thresholds, nb, color=F4_COLOR[k], lw=2.0, label=F4_LABEL[k].split(' (')[0])
    axC.plot(thresholds, nb_all, color='#555555', lw=1.6, ls='--', label='Treat all')
    axC.plot(thresholds, np.zeros_like(thresholds), color='#999999', lw=1.6, ls=':',
             label='Treat none')
    axC.set_xlim(0, 1); axC.set_ylim(-0.1, 0.5)
    axC.set_xlabel('Threshold probability', fontsize=12, fontweight='bold')
    axC.set_ylabel('Net benefit', fontsize=12, fontweight='bold')
    axC.legend(loc='upper right', fontsize=7.6, framealpha=0.88)
    axC.tick_params(labelsize=10)
    axC.grid(alpha=0.3, ls=':')
    axC.text(-0.18, 1.03, 'C', transform=axC.transAxes, fontsize=17, fontweight='bold')

    save(fig, 'Figure 4')

### `figureS6()`

In [ ]:
def figureS6():
    algo = S2['algo_res']
    colors = {'Logistic Regression': '#B2182B', 'Gradient Boosting': '#339C62',
              'SVM': '#E69F00', 'Gaussian Naive Bayes': '#8649BE'}
    fig, ax = plt.subplots(figsize=(2994 / 300, 2293 / 300))
    fig.subplots_adjust(left=0.09, right=0.99, top=0.99, bottom=0.11)
    ax.plot([0, 1], [0, 1], color='#999999', lw=1.5, ls='--')
    for name, res in algo.items():
        name = name.replace('Linear SVM', 'SVM')
        fpr, tpr, _ = roc_curve(y, res['p_patient'])
        auc_p = res['pooled_point']['auc']
        lo, hi = res['boot_ci']['auc']
        ax.plot(fpr, tpr, color=colors[name], lw=2.5,
                label=f'{name}: AUROC {auc_p:.3f} (95% CI, {lo:.3f}-{hi:.3f})')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel('1 - Specificity', fontsize=14, fontweight='bold')
    ax.set_ylabel('Sensitivity', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(alpha=0.3, ls=':')
    save(fig, 'Supplementary Figure S6')


# ============================================================ Supplementary Figure S7

### `figureS7()`

In [ ]:
def figureS7():
    """Supplementary Figure S7: among the 43 patients without SIC on Day 1, the cross-validated
    predicted-risk distribution by Day-3 status; individual points + median/IQR, descriptive."""
    sub = S2['subgroup']
    d1s = pd.read_csv(os.path.join(ROOT, 'data', 'NE_baseline_D1.csv'), encoding='utf-8-sig')
    d1s = d1s[d1s['Code'].isin(codes)].sort_values('Code').reset_index(drop=True)
    m = (d1s['SIC_D1'] == 'NSIC').values
    ys = y[m]

    models = [('Clinical', 'Clinical_Only'),
              ('Combined', 'Clinical_Immature'),
              ('Simplified', 'Parsimonious_Pipeline')]
    C_NO, C_YES = '#1A4A8C', '#D6604D'
    C_YES_LINE = '#B2182B'
    rng = np.random.default_rng(42)

    fig, ax = plt.subplots(figsize=(2120 / 300, 1760 / 300))
    fig.subplots_adjust(left=0.125, right=0.985, top=0.885, bottom=0.115)

    for i, (short, key) in enumerate(models):
        p = np.asarray(R['summary'][key]['p_patient'])[m]
        for grp, xoff, col, jit, size, alpha in [
                (0, -0.19, C_NO, 0.075, 24, 0.40),
                (1, +0.19, C_YES, 0.045, 46, 0.90)]:
            vals = p[ys == grp]
            if vals.size == 0:
                continue
            # median / IQR (drawn beneath the points)
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
            xc = i + 1 + xoff
            line_c = C_YES_LINE if grp == 1 else col
            ax.plot([xc, xc], [q1, q3], color=line_c, lw=1.6, alpha=0.9, zorder=2,
                    solid_capstyle='butt')
            ax.plot([xc - 0.10, xc + 0.10], [med, med], color=line_c, lw=3.0, zorder=2,
                    solid_capstyle='round')
            # individual points
            x = xc + rng.uniform(-jit, jit, vals.size)
            ax.scatter(x, vals, s=size, color=col, alpha=alpha, edgecolor='white',
                       linewidth=0.45, zorder=3)

    ax.set_xticks([i + 1 for i in range(len(models))])
    ax.set_xticklabels([mm[0] for mm in models], fontsize=12, fontweight='bold', color='#333333')
    ax.set_xlim(0.45, len(models) + 0.55)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Predicted probability of Day-3 SIC', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, ls=':')
    ax.set_axisbelow(True)

    from matplotlib.lines import Line2D as _Line2D
    ax.legend(handles=[
        _Line2D([], [], marker='o', ls='', ms=6, markerfacecolor=C_NO, markeredgecolor='white',
                alpha=0.6, label='No Day-3 SIC (n = %d)' % (sub['n'] - sub['events'])),
        _Line2D([], [], marker='o', ls='', ms=8, markerfacecolor=C_YES, markeredgecolor='white',
                label='Day-3 SIC (n = %d)' % sub['events']),
        _Line2D([], [], color='#555555', lw=1.6, label='Median (IQR)')],
        loc='upper left', fontsize=9, framealpha=0.92)

    save(fig, 'Supplementary Figure S7')



## Run (~1 min)

In [ ]:
from IPython.display import Image as IPyImage, display
figure3()
figure4()
figureS6()
figureS7()
for fign in ['Figure 3', 'Figure 4', 'Supplementary Figure S6', 'Supplementary Figure S7']:
    display(IPyImage(os.path.join(OUT_F, fign + '.png')))